# 競賽二 Transformer 訓練

執行 `aicup/tf_run.py` 的 final 模式：訓練池加全部 val648 玩家，3 個訓練 seed 依序訓練，寫 `final-ts{seed}.pt` 與 `final-ts{seed}-test.npy`。

- 打包資料先從 Drive 複製到 `/content`，輸出寫到 Drive 的 `humansl-analysis/tf-final-<名稱>/out/`
- 已存在的輸出會跳過，中斷後重跑同一個 MODE 會接續，接續時 `COMMIT` 要和開始時相同
- `tf_run.py` 只依賴 numpy 與 PyTorch，不 import repo 其他程式

| MODE | 訓練資料 | 競賽二最終版本 |
|---|---|---|
| `smoke` | 同 `j30_p4all`，每個 epoch 只跑 20 個 batch、1 個 seed | |
| `j30_p4all` | 6d 池（train 池、newdev）與 2d 池（train_p1～p3、train_p4 加 train_p4b）合訓，30 epoch | 最終版本的 TF |
| `6d` | 只用 6d 池 | 不在最終版本流程內 |
| `2d` | 只用 2d 的 train_p1、train_p2 | 不在最終版本流程內 |

- 最終提交檔由本機的 `aicup.blend_submit` 把 `j30_p4all` 的 test 機率和競賽二的 A、B 模型平均後寫出。

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下，在本機產生後上傳。打包的輸入是 `colab_humansl.ipynb` 寫出的 jsonl。

| 路徑 | MODE | 產生方式（本機，repo 根目錄） |
|---|---|---|
| `humansl-analysis/packed-moves/{trainpool,newdev,val648,test}_moves.npz` | `smoke`、`j30_p4all`、`6d` | `python -m aicup.pack_moves --6d`，輸入是 `colab_humansl` 的 `trainpool_hist`、`newdev_hist`、`val648_hist`、`test_hist` 與各自的 `_opp` |
| `humansl-analysis/packed-moves-2d/{train_p1,train_p2,val648,test}_moves.npz` | `smoke`、`j30_p4all`、`2d` | `python -m aicup.pack_moves`，輸入是 `colab_humansl` 的 `train_p1_2d`、`train_p2_2d`、`val648_2d`、`test_2d` 與各自的 `_opp` |
| `humansl-analysis/packed-moves-2d/{train_p3,train_p4all}_moves.npz` | `smoke`、`j30_p4all` | `python -m aicup.pack_moves train_p3 train_p4all`，輸入是 `train_p3_2d`、`train_p4_2d`、`train_p4b_2d` 與各自的 `_opp` |

## 參數

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'smoke'
AUTO_UNASSIGN = True

HA = 'humansl-analysis/'
POOL_6D = ['trainpool_moves.npz', 'newdev_moves.npz']
POOL_2D = ['train_p1_moves.npz', 'train_p2_moves.npz']
POOL_2D_ALL = [*POOL_2D, 'train_p3_moves.npz', 'train_p4all_moves.npz']
EVAL = ['val648_moves.npz', 'test_moves.npz']
JOINT = ['--joint', '--data2', '/content/data2', '--pool2', *POOL_2D_ALL]
RUNS = {  # MODE: (6d 或單池的訓練檔, 2d 池的訓練檔, 輸出名稱, 額外參數)
    'smoke': (POOL_6D, POOL_2D_ALL, 'smoke', [*JOINT, '--epochs', '1', '--train-seeds', '0', '--limit-batches', '20']),
    'j30_p4all': (POOL_6D, POOL_2D_ALL, 'j30-p4all', [*JOINT, '--epochs', '30', '--train-seeds', '0', '1', '2']),
    '6d': (POOL_6D, [], '6d', ['--epochs', '30', '--train-seeds', '0', '1', '2']),
    '2d': (POOL_2D, [], '2d', ['--epochs', '30', '--train-seeds', '0', '1', '2']),
}
if MODE not in RUNS:
    raise SystemExit('MODE 要是 %s 之一' % ', '.join(RUNS))
POOL, POOL2, NAME, EXTRA = RUNS[MODE]
SRC = HA + ('packed-moves-2d/' if MODE == '2d' else 'packed-moves/')
COPIES = [(SRC + f, '/content/data/' + f) for f in POOL + EVAL]
COPIES += [(HA + 'packed-moves-2d/' + f, '/content/data2/' + f) for f in (POOL2 + EVAL if POOL2 else [])]
OUT = HA + 'tf-final-%s/out' % NAME
CMD = ['python', 'aicup/tf_run.py', '--mode', 'final', '--loss', 'ce', '--data', '/content/data', '--pool', *POOL,
       *EXTRA, '--out', OUT]
HINTS = {'packed-moves/': 'python -m aicup.pack_moves --6d',
         'packed-moves-2d/': 'python -m aicup.pack_moves val648 train_p1 train_p2 test train_p3 train_p4all'}
print(' '.join(CMD))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號與記憶體。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 準備資料

要看到：需要的檔案都在、複製到 `/content/data`（與 `/content/data2`）的檔案清單。輸出目錄已用別的 commit 訓練過時停止。

In [ ]:
import shutil

link_drive(['humansl-analysis'])
need([(src, HINTS[src.split('/')[1] + '/']) for src, _ in COPIES])
check_commit(OUT, GIT_HEAD)
for src, dst in COPIES:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst)
    print('複製', src, '->', dst, '%.0f MB' % (os.path.getsize(dst) / 1e6))

## 執行

輸出同時寫到 Drive 的 `<輸出目錄>/final.log`。

In [ ]:
run(CMD, os.path.join(OUT, 'final.log'))

## 檢查

要看到：每個 seed 的 `final-ts{seed}-test.npy` 是 (10000, 10)，每列和為 1。

In [ ]:
import glob
import numpy as np

i = EXTRA.index('--train-seeds') + 1
seeds = []
while i < len(EXTRA) and not EXTRA[i].startswith('--'):
    seeds.append(int(EXTRA[i]))
    i += 1
for s in seeds:
    p = os.path.join(OUT, 'final-ts%d-test.npy' % s)
    if os.path.exists(p):
        x = np.load(p)
        print('%s %s，每列和 %.4f～%.4f' % (os.path.basename(p), x.shape, x.sum(1).min(), x.sum(1).max()))
    else:
        print('缺少', p)
print(sorted(os.path.basename(p) for p in glob.glob(os.path.join(OUT, '*'))))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')